# Week 3: In-class warm-up problems

**Course:** AI for Aerospace & Mechanical Engineering (PhD), Block 0
**Format:** 4 problems, 5 to 10 minutes each, pen and paper first, then verify in Python.

> **Provenance:** new content. Dataclass behaviour rewritten from
> [W3Schools dataclasses Module](https://www.w3schools.com/python/ref_module_dataclasses.asp) and
> [GeeksforGeeks Understanding Python Dataclasses](https://www.geeksforgeeks.org/python/understanding-python-dataclasses/);
> abstract classes from [GeeksforGeeks Abstract Classes in Python](https://www.geeksforgeeks.org/python/abstract-classes-in-python/);
> the Hampel filter adapted from Pearson (IEEE TCST, 2002).

## Problem 1 (5 min): what did `@dataclass` write for you?

<div dir="rtl">مسئله ۱: بدون اجرای کد، خروجی هر سطر را پیش‌بینی کنید. سپس توضیح دهید چرا <code>hash(a)</code> خطا می‌دهد و چه چیزی آن را درست می‌کند.</div>

```python
from dataclasses import dataclass, field

@dataclass
class Stage:
    name: str
    window: int = 5
    tags: list[str] = field(default_factory=list)
    mean_: float = field(default=0.0, repr=False, compare=False)

a = Stage("smooth")
b = Stage("smooth", 5)
b.mean_ = 3.2
a.tags.append("baro")
print(a)                                   # line 1
print(a == b, b.tags)                      # line 2
print(a == Stage("smooth", 5, ["baro"]))   # line 3
hash(a)                                    # line 4
```

a. Predict lines 1 to 3. b. Line 4 raises; which error, and why? c. What happens if
`tags: list[str] = []` is written instead of `default_factory`? d. Why is `mean_` declared with
`compare=False`?

## Problem 2 (10 min): the Hampel filter by hand

<div dir="rtl">مسئله ۲: فیلتر Hampel را با پنجره‌ی ۵ و آستانه‌ی ۳ روی نمونه‌های ۳ و ۴ دستی اجرا کنید. سپس بگویید چرا پنجره‌ی ۳ یک جهش دونمونه‌ای را نمی‌بیند.</div>

Airspeed samples (m/s): $v = [2.0,\ 2.1,\ 1.9,\ 2.0,\ 9.0,\ 2.2,\ 2.1]$, $k = 0 \dots 6$.
Rule: outlier if $\lvert v_k - m_k\rvert > 3 \times 1.4826 \times \text{MAD}_k$, with the median
$m_k$ and the MAD of the centred window of 5 samples.

a. Is $v_4 = 9.0$ an outlier? What replaces it?
b. Is $v_3 = 2.0$ an outlier?
c. A different record has a **two-sample** spike, $[2.0,\ 2.1,\ 9.0,\ 9.1,\ 2.0,\ 2.1,\ 1.9]$.
   Test $k=2$ with a window of 3 and with a window of 5. What rule for the window length follows?

## Problem 3 (10 min): does the order of two stages matter?

<div dir="rtl">مسئله ۳: یک جهش تک‌نمونه‌ای را یک بار با «اول حذف جهش، بعد میانگین» و یک بار با «اول میانگین، بعد حذف جهش» پردازش کنید و بیشینه‌ی خروجی را مقایسه کنید.</div>

Signal $x = [1, 1, 1, 10, 1, 1, 1]$ (a +9 glitch at $k=3$). $H$: Hampel filter, window 3,
$n_\sigma = 3$. $S$: centred 3-sample moving average, truncated at the ends.

a. Compute $S(H(x))$ and its maximum.
b. Compute $H(S(x))$ and its maximum.
c. A structures engineer needs the peak load factor. Which order is right, and what is the
   relative error of the other one if the true signal is 1?
d. How does `DataPipeline` make the order visible and comparable?

## Problem 4 (10 min): which pipelines can be built, and when does each error appear?

<div dir="rtl">مسئله ۴: با توجه به نوع ورودی و خروجی هر مرحله، بگویید کدام خط لوله ساخته می‌شود و خطای هر کدام در «زمان ساخت» رخ می‌دهد یا «زمان اجرا».</div>

| Stage | `INPUT_TYPE` → `OUTPUT_TYPE` | learns? |
|-------|------------------------------|---------|
| `SensorToFrame` | `BaseSensor` → `DataFrame` | no |
| `HampelFilter`, `RollingMean`, `SelectColumns` | `DataFrame` → `DataFrame` | no |
| `Standardize` | `DataFrame` → `DataFrame` | yes |
| `ToArray` | `DataFrame` → `ndarray` | no |

For each line say: built or not; if not, the error and its message; if built, what happens on
`p(data)`.

1. `p = SensorToFrame() >> HampelFilter(["x"]) >> ToArray(["x"])`, then `p(imu)`.
2. `p = HampelFilter(["x"]) >> SensorToFrame()`.
3. `p = ToArray(["x"]) >> Standardize(["x"])`.
4. `p = SelectColumns(["x"]) >> Standardize(["x"]) >> ToArray(["x"])`, then `p(frame)` without `fit`.
5. `class Despiker(PipelineStage): pass`, then `Despiker()`.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
# Scratch space for verifying your hand calculations, e.g.
# from aerospace_ai.pipeline import HampelFilter, RollingMean